In [4]:
import torch
import tomllib

with open("path.toml", "rb") as f:
    config = tomllib.load(f)

## 文件路径表
file_path_list = config['required_files']
## 字典文件路径
csv_path = config['csv_files']['RFT']

print(file_path_list,csv_path)

{'FVs_biomed': '//cabinet/derivatives/DeepDR/FVs_biomed', 'FVs_ConceptCLIP': '//cabinet/derivatives/DeepDR/FVs_ConceptCLIP', 'FVs_CONCH': '//cabinet/derivatives/DeepDR/FVs_CONCH', 'FVs_CXRCLIP': '//cabinet/derivatives/DeepDR/FVs_CXRCLIP', 'FVs_DINOv2': '//cabinet/derivatives/DeepDR/FVs_DINOv2', 'FVs_EVA02-L-14-336': '//cabinet/derivatives/DeepDR/FVs_EVA02-L-14-336', 'FVs_FLAIR': '//cabinet/derivatives/DeepDR/FVs_FLAIR', 'FVs_INViT-L-16': '//cabinet/derivatives/DeepDR/FVs_INViT-L-16', 'FVs_llava-med': '//cabinet/derivatives/DeepDR/FVs_llava-med', 'FVs_llava-Mistral-7b': '//cabinet/derivatives/DeepDR/FVs_llava-Mistral-7b', 'FVs_llava-vicuna-13b-hf': '//cabinet/derivatives/DeepDR/FVs_llava-vicuna-13b-hf', 'FVs_LLM2CLIP-EVA': '//cabinet/derivatives/DeepDR/FVs_LLM2CLIP-EVA', 'FVs_LLM2CLIP-openai': '//cabinet/derivatives/DeepDR/FVs_LLM2CLIP-openai', 'FVs_medsiglip': '//cabinet/derivatives/DeepDR/FVs_medsiglip', 'FVs_MedTrinity': '//cabinet/derivatives/DeepDR/FVs_MedTrinity', 'FVs_MONET': '//

In [2]:
file_name = 'FVs_biomed'
file_path = file_path_list['FVs_biomed']
print(file_path)
data = torch.load(file_path, map_location='cpu', weights_only=False)


//cabinet/derivatives/DeepDR/FVs_biomed


In [ ]:
import pandas as pd
import numpy as np

ann = pd.read_csv(csv_path)

# 字符串预处理
ann.columns = ann.columns.str.strip()
ann["image_id"] = ann["image_id"].astype(str).str.strip()

print(ann.shape)
print(ann.columns.tolist())
print(ann.head())

(1200, 10)
['patient_id', 'image_id', 'image_path', 'Overall quality', 'left_eye_DR_Level', 'right_eye_DR_Level', 'patient_DR_Level', 'Clarity', 'Field definition', 'Artifact']
   patient_id image_id                           image_path  Overall quality  \
0           1     1_l1  \regular-fundus-training\1\1_l1.jpg                0   
1           1     1_l2  \regular-fundus-training\1\1_l2.jpg                0   
2           1     1_r1  \regular-fundus-training\1\1_r1.jpg                0   
3           1     1_r2  \regular-fundus-training\1\1_r2.jpg                0   
4           2     2_l1  \regular-fundus-training\2\2_l1.jpg                0   

   left_eye_DR_Level  right_eye_DR_Level  patient_DR_Level  Clarity  \
0                0.0                 NaN                 0        8   
1                0.0                 NaN                 0        8   
2                NaN                 0.0                 0        8   
3                NaN                 0.0                 0

In [6]:
##绑定 数据和annotation
def to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)

feature_dict = {}

for k, v in data.items():
    try:
        arr = to_numpy(v)
        if arr.ndim == 1:
            feature_dict[str(k).strip()] = arr.astype(np.float32)
    except Exception:
        pass

print("特征数量:", len(feature_dict))
print("前10个key:", list(feature_dict.keys())[:10])
print("单个特征shape:", next(iter(feature_dict.values())).shape)

特征数量: 1200
前10个key: ['1_l1', '1_l2', '1_r1', '1_r2', '2_l1', '2_l2', '2_r1', '2_r2', '3_l1', '3_l2']
单个特征shape: (512,)


In [7]:
## 测试对齐状况

csv_ids = set(ann["image_id"])
feature_ids = set(feature_dict.keys())

print("CSV中的image数量:", len(csv_ids))
print("特征字典中的image数量:", len(feature_ids))

print("CSV有但特征没有:", len(csv_ids - feature_ids))
print(list(csv_ids - feature_ids)[:20])

print("特征有但CSV没有:", len(feature_ids - csv_ids))
print(list(feature_ids - csv_ids)[:20])

CSV中的image数量: 1200
特征字典中的image数量: 1200
CSV有但特征没有: 0
[]
特征有但CSV没有: 0
[]


In [8]:
# 只保留有特征的样本
meta_df = ann[ann["image_id"].isin(feature_dict.keys())].copy()

# 保持一个固定顺序
meta_df = meta_df.sort_values(["patient_id", "image_id"]).reset_index(drop=True)

# 构造特征矩阵 X
X = np.stack([feature_dict[img_id] for img_id in meta_df["image_id"]])

print("X shape:", X.shape)
print("meta_df shape:", meta_df.shape)

assert X.shape[0] == meta_df.shape[0]

X shape: (1200, 512)
meta_df shape: (1200, 10)


In [9]:
# 合并左右眼标签
# 解析左右眼
meta_df["eye"] = meta_df["image_id"].str.extract(r"_(l|r)")[0]

# 转成数值
for col in [
    "left_eye_DR_Level",
    "right_eye_DR_Level",
    "patient_DR_Level",
    "Overall quality",
    "Clarity",
    "Field definition",
    "Artifact"
]:
    meta_df[col] = pd.to_numeric(meta_df[col], errors="coerce")

# 左眼图像用 left_eye_DR_Level，右眼图像用 right_eye_DR_Level
meta_df["image_DR_Level"] = np.where(
    meta_df["eye"] == "l",
    meta_df["left_eye_DR_Level"],
    meta_df["right_eye_DR_Level"]
)

print(meta_df[[
    "patient_id",
    "image_id",
    "eye",
    "image_DR_Level",
    "patient_DR_Level",
    "Overall quality",
    "Clarity",
    "Field definition",
    "Artifact"
]].head(10))

print(meta_df["image_DR_Level"].value_counts(dropna=False).sort_index())

   patient_id image_id eye  image_DR_Level  patient_DR_Level  Overall quality  \
0           1     1_l1   l             0.0                 0                0   
1           1     1_l2   l             0.0                 0                0   
2           1     1_r1   r             0.0                 0                0   
3           1     1_r2   r             0.0                 0                0   
4           2     2_l1   l             2.0                 2                0   
5           2     2_l2   l             2.0                 2                0   
6           2     2_r1   r             2.0                 2                0   
7           2     2_r2   r             2.0                 2                0   
8           3     3_l1   l             1.0                 1                0   
9           3     3_l2   l             1.0                 1                0   

   Clarity  Field definition  Artifact  
0        8                 8         4  
1        8                

In [10]:
#保存
save_name = f"aligned_{file_name}.npz"

np.savez(
    save_name,
    X=X,
    image_id=meta_df["image_id"].values,
    patient_id=meta_df["patient_id"].values,
    eye=meta_df["eye"].values,
    image_DR_Level=meta_df["image_DR_Level"].values,
    patient_DR_Level=meta_df["patient_DR_Level"].values,
    overall_quality=meta_df["Overall quality"].values,
    clarity=meta_df["Clarity"].values,
    field_definition=meta_df["Field definition"].values,
    artifact=meta_df["Artifact"].values
)

meta_df.to_csv(f"aligned_{file_name}_metadata.csv", index=False)

print("保存完成:", save_name)

保存完成: aligned_FVs_biomed.npz


In [16]:
import os
import numpy as np
import pandas as pd
import torch

def to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def load_feature_dict(file_path):
    data = torch.load(file_path, map_location="cpu", weights_only=False)

    feature_dict = {}
    for k, v in data.items():
        try:
            arr = to_numpy(v)
            if arr.ndim == 1:
                feature_dict[str(k).strip()] = arr.astype(np.float32)
        except Exception:
            pass

    return feature_dict


def align_and_save(
    file_name,
    file_path,
    ann,
    save_dir="./aligned_features_drvalid",
    drop_missing_dr=True
):
    os.makedirs(save_dir, exist_ok=True)

    feature_dict = load_feature_dict(file_path)

    ann = ann.copy()
    ann.columns = ann.columns.str.strip()
    ann["image_id"] = ann["image_id"].astype(str).str.strip()

    # 只保留有特征向量的样本
    meta_df = ann[ann["image_id"].isin(feature_dict.keys())].copy()
    meta_df = meta_df.sort_values(["patient_id", "image_id"]).reset_index(drop=True)

    # 解析左右眼
    meta_df["eye"] = meta_df["image_id"].str.extract(r"_(l|r)")[0]

    # 转数值
    for col in [
        "left_eye_DR_Level",
        "right_eye_DR_Level",
        "patient_DR_Level",
        "Overall quality",
        "Clarity",
        "Field definition",
        "Artifact"
    ]:
        meta_df[col] = pd.to_numeric(meta_df[col], errors="coerce")

    # 更安全的写法：左眼图像取 left_eye_DR_Level，右眼图像取 right_eye_DR_Level
    meta_df["eye_DR_Level"] = np.nan

    meta_df.loc[meta_df["eye"] == "l", "eye_DR_Level"] = meta_df.loc[
        meta_df["eye"] == "l", "left_eye_DR_Level"
    ]

    meta_df.loc[meta_df["eye"] == "r", "eye_DR_Level"] = meta_df.loc[
        meta_df["eye"] == "r", "right_eye_DR_Level"
    ]

    # 保存被删除的缺失标签样本，方便报告里说明
    missing_df = meta_df[meta_df["eye_DR_Level"].isna()].copy()
    missing_path = f"{save_dir}/removed_missing_DR_{file_name}.csv"
    missing_df.to_csv(missing_path, index=False)

    # 如果用于DR grade分析，删除缺失DR标签的样本
    if drop_missing_dr:
        meta_df = meta_df.dropna(subset=["eye_DR_Level"]).copy()
        meta_df = meta_df.reset_index(drop=True)

    # 现在再构造 X，保证 X 和 meta_df 完全对齐
    X = np.stack([feature_dict[img_id] for img_id in meta_df["image_id"]])

    # 保存路径
    suffix = "drvalid" if drop_missing_dr else "all"
    save_path = f"{save_dir}/aligned_{suffix}_{file_name}.npz"
    meta_path = f"{save_dir}/aligned_{suffix}_{file_name}_metadata.csv"

    np.savez(
        save_path,
        X=X,
        image_id=meta_df["image_id"].values,
        patient_id=meta_df["patient_id"].values,
        eye=meta_df["eye"].values,
        eye_DR_Level=meta_df["eye_DR_Level"].values,
        patient_DR_Level=meta_df["patient_DR_Level"].values,
        overall_quality=meta_df["Overall quality"].values,
        clarity=meta_df["Clarity"].values,
        field_definition=meta_df["Field definition"].values,
        artifact=meta_df["Artifact"].values
    )

    meta_df.to_csv(meta_path, index=False)

    print("=" * 60)
    print(file_name)
    print("feature数量:", len(feature_dict))
    print("保存模式:", suffix)
    print("删除缺失DR标签数:", len(missing_df) if drop_missing_dr else 0)
    print("对齐后X:", X.shape)
    print("DR标签分布:")
    print(meta_df["eye_DR_Level"].value_counts().sort_index())
    print("保存:", save_path)
    print("metadata:", meta_path)
    print("removed missing:", missing_path)

    return save_path, meta_path
#废弃版本*********************************************************
#  ##对所有文件都做这个操作

# import numpy as np
# import pandas as pd
# import torch

# def to_numpy(x):
#     if torch.is_tensor(x):
#         return x.detach().cpu().numpy()
#     return np.asarray(x)

# def load_feature_dict(file_path):
#     data = torch.load(file_path, map_location="cpu", weights_only=False)
    
#     feature_dict = {}
#     for k, v in data.items():
#         try:
#             arr = to_numpy(v)
#             if arr.ndim == 1:
#                 feature_dict[str(k).strip()] = arr.astype(np.float32)
#         except Exception:
#             pass
    
#     return feature_dict

# def align_and_save(file_name, file_path, ann, save_dir="./aligned_features"):
#     import os
#     os.makedirs(save_dir, exist_ok=True)

#     feature_dict = load_feature_dict(file_path)

#     ann = ann.copy()
#     ann["image_id"] = ann["image_id"].astype(str).str.strip()

#     meta_df = ann[ann["image_id"].isin(feature_dict.keys())].copy()
#     meta_df = meta_df.sort_values(["patient_id", "image_id"]).reset_index(drop=True)

#     X = np.stack([feature_dict[img_id] for img_id in meta_df["image_id"]])

#     meta_df["eye"] = meta_df["image_id"].str.extract(r"_(l|r)")[0]

#     for col in [
#         "left_eye_DR_Level",
#         "right_eye_DR_Level",
#         "patient_DR_Level",
#         "Overall quality",
#         "Clarity",
#         "Field definition",
#         "Artifact"
#     ]:
#         meta_df[col] = pd.to_numeric(meta_df[col], errors="coerce")

#     meta_df["eye_DR_Level"] = np.where(
#         meta_df["eye"] == "l",
#         meta_df["left_eye_DR_Level"],
#         meta_df["right_eye_DR_Level"]
#     )

#     save_path = f"{save_dir}/aligned_{file_name}.npz"
#     meta_path = f"{save_dir}/aligned_{file_name}_metadata.csv"

#     np.savez(
#         save_path,
#         X=X,
#         image_id=meta_df["image_id"].values,
#         patient_id=meta_df["patient_id"].values,
#         eye=meta_df["eye"].values,
#         eye_DR_Level=meta_df["eye_DR_Level"].values,
#         patient_DR_Level=meta_df["patient_DR_Level"].values,
#         overall_quality=meta_df["Overall quality"].values,
#         clarity=meta_df["Clarity"].values,
#         field_definition=meta_df["Field definition"].values,
#         artifact=meta_df["Artifact"].values
#     )

#     meta_df.to_csv(meta_path, index=False)

#     print("=" * 60)
#     print(file_name)
#     print("feature数量:", len(feature_dict))
#     print("对齐后X:", X.shape)
#     print("缺失DR标签:", meta_df["eye_DR_Level"].isna().sum())
#     print("保存:", save_path)

#     return save_path, meta_path

In [17]:
# ann 是你读入的 annotation 表
# file_path_list 是你从 path.toml 读出来的路径字典

# 版本1：保留全部1200
for file_name, file_path in file_path_list.items():
    if file_name.startswith("FVs_"):
        align_and_save(
            file_name=file_name,
            file_path=file_path,
            ann=ann,
            save_dir="./aligned_features_all",
            drop_missing_dr=False
        )
# 版本2：只保留有DR标签的1189
for file_name, file_path in file_path_list.items():
    if file_name.startswith("FVs_"):
        align_and_save(
            file_name=file_name,
            file_path=file_path,
            ann=ann,
            save_dir="./aligned_features_drvalid",
            drop_missing_dr=True
        )


FVs_biomed
feature数量: 1200
保存模式: all
删除缺失DR标签数: 0
对齐后X: (1200, 512)
DR标签分布:
eye_DR_Level
0.0    532
1.0    139
2.0    232
3.0    214
4.0     72
Name: count, dtype: int64
保存: ./aligned_features_all/aligned_all_FVs_biomed.npz
metadata: ./aligned_features_all/aligned_all_FVs_biomed_metadata.csv
removed missing: ./aligned_features_all/removed_missing_DR_FVs_biomed.csv
FVs_ConceptCLIP
feature数量: 1200
保存模式: all
删除缺失DR标签数: 0
对齐后X: (1200, 1152)
DR标签分布:
eye_DR_Level
0.0    532
1.0    139
2.0    232
3.0    214
4.0     72
Name: count, dtype: int64
保存: ./aligned_features_all/aligned_all_FVs_ConceptCLIP.npz
metadata: ./aligned_features_all/aligned_all_FVs_ConceptCLIP_metadata.csv
removed missing: ./aligned_features_all/removed_missing_DR_FVs_ConceptCLIP.csv
FVs_CONCH
feature数量: 1200
保存模式: all
删除缺失DR标签数: 0
对齐后X: (1200, 512)
DR标签分布:
eye_DR_Level
0.0    532
1.0    139
2.0    232
3.0    214
4.0     72
Name: count, dtype: int64
保存: ./aligned_features_all/aligned_all_FVs_CONCH.npz
metadata: ./aligned_fea

In [ ]:
## 错误分析

import pandas as pd

meta_df = pd.read_csv("./aligned_features/aligned_FVs_biomed_metadata.csv")

print(meta_df.columns.tolist())
print(meta_df.head())

missing = meta_df[meta_df["eye_DR_Level"].isna()].copy()

missing["image_index"] = missing["image_id"].str.extract(r"_(?:l|r)(\d+)")[0]

print(missing[[
    "patient_id",
    "image_id",
    "eye",
    "image_index",
    "left_eye_DR_Level",
    "right_eye_DR_Level",
    "patient_DR_Level",
    "Overall quality",
    "Clarity",
    "Field definition",
    "Artifact"
]].to_string(index=False))

print("缺失样本的 image_index 分布:")
print(missing["image_index"].value_counts(dropna=False))


#  一部の特殊な画像ID（例：l3, r3, l4, r4）では、image_id に含まれる左右情報と
#  left_eye_DR_Level / right_eye_DR_Level の非欠損列が一致しない例が確認された。
#  公式のデータ説明においてこの例外処理に関する明示的な記述は確認できなかったため、
#  本解析ではラベルの誤付与を避ける目的で、該当する11画像をDR grade解析から除外した。

['patient_id', 'image_id', 'image_path', 'Overall quality', 'left_eye_DR_Level', 'right_eye_DR_Level', 'patient_DR_Level', 'Clarity', 'Field definition', 'Artifact', 'eye', 'eye_DR_Level']
   patient_id image_id                           image_path  Overall quality  \
0           1     1_l1  \regular-fundus-training\1\1_l1.jpg                0   
1           1     1_l2  \regular-fundus-training\1\1_l2.jpg                0   
2           1     1_r1  \regular-fundus-training\1\1_r1.jpg                0   
3           1     1_r2  \regular-fundus-training\1\1_r2.jpg                0   
4           2     2_l1  \regular-fundus-training\2\2_l1.jpg                0   

   left_eye_DR_Level  right_eye_DR_Level  patient_DR_Level  Clarity  \
0                0.0                 NaN                 0        8   
1                0.0                 NaN                 0        8   
2                NaN                 0.0                 0        8   
3                NaN                 0.0      